# Mapeando Datos y Fijando Valores
## Dominando Diccionarios y Tuplas en Python

**Semana 09 - Fundamentos de Python**  
**Cuaderno de trabajo del estudiante**  
**Caso:** análisis de hasta 1,000,000 de pacientes sintéticos

### Meta de la sesión

Hoy no nos limitamos a ejecutar código: construimos funciones e índices de consulta. Al terminar podremos:

1. contar valores con diccionarios;
2. recorrer listas internas mediante ciclos anidados;
3. crear mapas que contienen otros mapas;
4. recorrer el conjunto una sola vez y consultar sus resúmenes muchas veces;
5. fijar resultados como tuplas `(categoría, cantidad)`.

> **Idea guía:** los registros son el detalle; los diccionarios serán nuestros índices de consulta.


## 1. Preparamos y cargamos los datos (0-15 min)

El generador usa una semilla fija. Los primeros registros incluyen nombres del grupo para reconocer el conjunto, pero **edades, diagnósticos, ubicaciones, medicamentos y síntomas son completamente sintéticos**.

Por defecto trabajamos con el millón. Para practicar con 10,000 registros cambiamos juntos `NOMBRE_ARCHIVO_TRABAJO` y `TOTAL_ESPERADO`; así conservamos el archivo principal y usamos un destino diferente.

> No abrimos el JSON completo en el editor. La carga del millón puede usar cerca de 1.5 GB de memoria.


In [1]:
from pathlib import Path
import sys

# Modo principal de la clase:
NOMBRE_ARCHIVO_TRABAJO = "clinica_s09.json"
TOTAL_ESPERADO = 1_000_000

# Modo de práctica para equipos con menos memoria (cambiamos ambas líneas):
# NOMBRE_ARCHIVO_TRABAJO = "clinica_s09_muestra_10000.json"
# TOTAL_ESPERADO = 10_000

carpeta_semana = Path.cwd()
if not (carpeta_semana / "generar_clinica_s09.py").exists():
    carpeta_semana = Path.cwd() / "Semana 09"

if not (carpeta_semana / "generar_clinica_s09.py").exists():
    raise FileNotFoundError("Abrimos el notebook desde la raíz del curso o desde Semana 09.")

sys.path.insert(0, str(carpeta_semana.resolve()))
#from generar_clinica_s09 import generar_archivo

ruta_datos = carpeta_semana / NOMBRE_ARCHIVO_TRABAJO
if not ruta_datos.exists():
    ruta_datos = generar_archivo(total=TOTAL_ESPERADO, destino=ruta_datos)

print(f"Datos disponibles: {ruta_datos} ({ruta_datos.stat().st_size / 1_000_000:.1f} MB)")


Datos disponibles: c:\Dev\sites-didacticos\2026C2-G01-FUNDAMENTOS DE PYTHON\Clase_09\clinica_s09.json (287.5 MB)


In [2]:
import json


def cargar_json(ruta):
    try:
        with ruta.open("r", encoding="utf-8") as archivo:
            return json.load(archivo)
    except FileNotFoundError as error:
        raise FileNotFoundError("Ejecutamos primero la celda de preparación.") from error
    except json.JSONDecodeError as error:
        raise ValueError("El JSON está incompleto. Lo generamos nuevamente.") from error
    except MemoryError as error:
        raise MemoryError("Cerramos otras aplicaciones, reiniciamos el kernel e intentamos de nuevo.") from error


pacientes = cargar_json(ruta_datos)
if not isinstance(pacientes, list):
    raise TypeError("El archivo de trabajo debe contener una lista de pacientes.")

if len(pacientes) != TOTAL_ESPERADO:
    print("El archivo no coincide con TOTAL_ESPERADO; lo generamos nuevamente.")
    ruta_datos = generar_archivo(total=TOTAL_ESPERADO, destino=ruta_datos)
    pacientes = cargar_json(ruta_datos)

if len(pacientes) != TOTAL_ESPERADO:
    raise ValueError(f"Esperábamos {TOTAL_ESPERADO:,} pacientes y recibimos {len(pacientes):,}.")

muestra = pacientes[:min(5_000, len(pacientes))]
print(f"Pacientes cargados: {len(pacientes):,} | Muestra de práctica: {len(muestra):,}")


Pacientes cargados: 1,000,000 | Muestra de práctica: 5,000


## Actividad 1 — Exploramos un registro

Ejecutamos siempre las celdas anteriores. La guardia siguiente transforma un `NameError` poco informativo en una instrucción concreta. Después producimos **5-7 líneas** para inspeccionar tipos, campos y un síntoma.


In [3]:
if "pacientes" not in globals():
    raise RuntimeError("Ejecutamos primero las celdas de preparación y carga.")

# Producimos aquí 5-7 líneas para seleccionar el primer registro e inspeccionar su estructura.
print("Lista:", type(pacientes).__name__)
print(f"Tamaño de la lsita: {len(pacientes):,.0f}")
primer_paciente = pacientes[0]
print("Primer paciente:", type(primer_paciente).__name__)  
print("Campos:", tuple(primer_paciente.keys()))
print("Síntomas:", type(primer_paciente["sintomas"]).__name__)
print("Primer síntoma primer paciente:", primer_paciente["sintomas"][0])
if isinstance (primer_paciente, dict):
    print(primer_paciente)
    

Lista: list
Tamaño de la lsita: 1,000,000
Primer paciente: dict
Campos: ('carnet', 'nombre', 'edad', 'genero', 'provincia', 'canton', 'visitas', 'enfermedad', 'medicamento', 'activo', 'sintomas')
Síntomas: list
Primer síntoma primer paciente: náuseas
{'carnet': 115000001, 'nombre': 'Germán Antonio Cerdas Valle', 'edad': 22, 'genero': 'M', 'provincia': 'Alajuela', 'canton': 'Alajuela', 'visitas': 22, 'enfermedad': 'alzheimer', 'medicamento': 'losartán', 'activo': True, 'sintomas': ['náuseas', 'fatiga', 'dolor muscular']}


In [4]:
from verificaciones_s09 import (
    verificar_contador,
    verificar_mapa_anidado,
    verificar_maximo,
)

print("Verificaciones de apoyo disponibles.")


Verificaciones de apoyo disponibles.


## Actividad 2 — Mapeamos y fijamos (15-35 min)

Producimos **8-10 líneas**. Recorremos `.items()`, creamos una tupla por turno, derivamos la tupla con mayor cantidad mediante el recorrido, la desempaquetamos y demostramos la inmutabilidad capturando `TypeError`.


In [5]:
pacientes_por_turno = {
    "mañana": 18, 
    "tarde": 25
    }
cantidad_mayor = 0
# Recorremos, derivamos la tupla mayor, desempaquetamos y comprobamos la inmutabilidad.
for turno, cantidad in pacientes_por_turno.items():
    if cantidad > cantidad_mayor:
        cantidad_mayor = cantidad
        turno_mayor = (turno, cantidad)
        
try:
    turno_mayor[0] = "Manana"
except TypeError:
    print("La tupla esta protegida y es mas rápida en lectura")
print("Turno con mas pacientes:", turno_mayor)

nums = [10,20,30,60,10,50,30]
mayor = 0
for n in nums:
    if n > mayor:
        mayor = n
print(mayor)

La tupla esta protegida y es mas rápida en lectura
Turno con mas pacientes: ('tarde', 25)
60


## Actividad 3 — Contamos cualquier campo

Construimos `contar_por_campo(registros, campo) -> dict`. Aplicamos el patrón:

```python
conteo[valor] = conteo.get(valor, 0) + 1
```

Probamos primero con `muestra`, no con el conjunto completo.


In [6]:
def contar_por_campo(registros, campo) -> dict:
    """Cuenta las apariciones de cada valor de un campo."""
    # Aplicamos aquí el patrón visible.
    conteo = {}
    for elemento in registros:
        valor = elemento[campo]
        conteo[valor] = conteo.get(valor , 0) + 1
    return conteo


enfermedades_muestra = contar_por_campo(muestra, "enfermedad")
print(enfermedades_muestra)

generos_pacientes = contar_por_campo(pacientes,'genero')
print(generos_pacientes)

conteo_provincia = contar_por_campo(pacientes,"provincia")
print(conteo_provincia)
verificar_contador(enfermedades_muestra, muestra, "enfermedad", "enfermedades de la muestra")


{'alzheimer': 118, 'arritmia': 133, 'esclerosis múltiple': 126, 'epilepsia': 133, 'depresión mayor': 134, 'hepatitis': 145, 'ansiedad generalizada': 119, 'otitis': 113, 'psoriasis': 133, 'asma': 134, 'conjuntivitis': 138, 'gota': 116, 'insuficiencia renal': 121, 'osteoporosis': 124, 'cistitis': 108, 'fibromialgia': 116, 'sinusitis': 115, 'migraña': 115, 'hipertiroidismo': 116, 'varicela': 134, 'gripe': 124, 'alergia': 146, 'hipertensión': 134, 'dermatitis': 137, 'trastorno bipolar': 135, 'estrés': 106, 'colitis': 101, 'artritis': 126, 'parkinson': 109, 'diabetes': 136, 'anemia': 139, 'obesidad': 131, 'úlcera péptica': 126, 'bronquitis': 132, 'lupus': 121, 'hipotiroidismo': 123, 'faringitis': 129, 'lumbalgia': 115, 'gastritis': 129, 'tiroiditis': 110}
{'M': 500786, 'F': 499214}
{'Alajuela': 143056, 'San José': 143581, 'Heredia': 142439, 'Cartago': 142818, 'Limón': 143092, 'Puntarenas': 142351, 'Guanacaste': 142663}
Correcto: enfermedades de la muestra contiene 40 claves y suma 5,000.


## Actividad 4 — Contamos listas internas 

Construimos `contar_sintomas(registros) -> dict` a partir de este pseudocódigo:

1. creamos el diccionario de conteos;
2. por cada registro, recorremos sus síntomas;
3. actualizamos el conteo de cada síntoma;
4. devolvemos el diccionario.

Cada registro tiene tres síntomas; la verificación deriva la suma esperada directamente de los datos.


In [7]:
def contar_sintomas(registros) -> dict:
    """Cuenta los síntomas contenidos en la lista interna de cada registro."""
    # Convertimos el pseudocódigo en Python.
    conteo = dict()
    for registro in registros:
        for sintoma in registro["sintomas"]:
            conteo[sintoma] = conteo.get(sintoma, 0) + 1
    return conteo


sintomas_muestra = contar_sintomas(muestra)
sintomas_muestra_pacientes = contar_sintomas(pacientes)

print(sintomas_muestra)
print(sintomas_muestra_pacientes)
verificar_contador(sintomas_muestra, muestra, "sintomas", "síntomas de la muestra", campo_lista=True)


{'náuseas': 209, 'fatiga': 224, 'dolor muscular': 233, 'confusión': 219, 'fiebre': 232, 'dificultad para tragar': 197, 'caída de cabello': 234, 'depresión': 243, 'vómitos': 225, 'mareos': 256, 'debilidad muscular': 224, 'visión borrosa': 232, 'irritabilidad': 241, 'dificultad para respirar': 208, 'dolor abdominal': 230, 'heridas que no sanan': 237, 'estornudos': 213, 'secreción nasal': 232, 'dolor al orinar': 237, 'dolor de cabeza': 231, 'picazón': 240, 'inflamación': 257, 'sudoración nocturna': 249, 'micción frecuente': 217, 'boca seca': 236, 'escalofríos': 248, 'calambres': 227, 'piel grasosa': 222, 'pérdida de apetito': 244, 'esputo amarillo': 223, 'erupciones': 225, 'insomnio': 244, 'zumbido en oídos': 242, 'moretones': 238, 'tos con sangre': 254, 'dolor articular': 232, 'sensibilidad al sonido': 252, 'palpitaciones': 243, 'hormigueo': 223, 'ictericia': 220, 'manchas en la piel': 230, 'dolor de garganta': 250, 'dolor lumbar': 266, 'convulsiones': 244, 'ojos rojos': 229, 'sed excesi

## Pausa 

Antes de continuar, explicamos en parejas:

- cuál ciclo entra primero y cuál después al contar síntomas;
- por qué `.get(clave, 0)` evita un error con una clave nueva.


## Actividad 5 — Mapeamos provincia → enfermedad 
Trabajamos ahora sin patrón ni pseudocódigo. Construimos:

```python
contar_enfermedades_por_provincia(registros) -> dict
```

Representamos el resultado como `{provincia: {enfermedad: cantidad}}` y lo probamos sobre la muestra.


In [8]:
def contar_enfermedades_por_provincia(registros) -> dict:
    """Agrupa conteos de enfermedades dentro de cada provincia."""
    # Resolvemos el contrato sin plantilla de algoritmo.
    mapa = {}
    for registro in registros:
        provincia = registro['provincia'] 
        enfermedad = registro['enfermedad']
        if provincia not in mapa:
            mapa[provincia] = {}
        conteos_provincia = mapa[provincia]
        conteos_provincia[enfermedad] = conteos_provincia.get(enfermedad , 0) + 1    
    return mapa



provincias_muestra = contar_enfermedades_por_provincia(muestra)
print(provincias_muestra)
verificar_mapa_anidado(
    provincias_muestra, muestra, "provincia", "enfermedad", "mapa provincial de la muestra"
)


{'Alajuela': {'alzheimer': 16, 'arritmia': 16, 'hepatitis': 12, 'otitis': 15, 'gota': 14, 'osteoporosis': 14, 'alergia': 24, 'dermatitis': 17, 'hipertiroidismo': 11, 'psoriasis': 22, 'lupus': 20, 'asma': 25, 'esclerosis múltiple': 22, 'epilepsia': 22, 'lumbalgia': 16, 'varicela': 23, 'sinusitis': 16, 'faringitis': 14, 'migraña': 21, 'hipertensión': 21, 'anemia': 13, 'colitis': 11, 'hipotiroidismo': 12, 'diabetes': 21, 'tiroiditis': 15, 'trastorno bipolar': 18, 'úlcera péptica': 20, 'obesidad': 15, 'insuficiencia renal': 21, 'fibromialgia': 17, 'artritis': 18, 'ansiedad generalizada': 21, 'parkinson': 18, 'cistitis': 18, 'gastritis': 22, 'gripe': 12, 'bronquitis': 15, 'conjuntivitis': 18, 'depresión mayor': 20, 'estrés': 15}, 'San José': {'alzheimer': 14, 'esclerosis múltiple': 22, 'ansiedad generalizada': 16, 'otitis': 17, 'migraña': 15, 'sinusitis': 17, 'hipertensión': 20, 'dermatitis': 20, 'lupus': 13, 'psoriasis': 13, 'obesidad': 18, 'insuficiencia renal': 17, 'arritmia': 23, 'hipot

## Actividad 6 — Integramos índices en un recorrido 

Construimos `construir_indices(registros) -> tuple[dict, dict, dict]` bajo este contrato:

- devolvemos, en ese orden, los conteos de enfermedades, síntomas y enfermedades por provincia;
- recorremos `registros` con un único ciclo externo;
- conservamos todas las categorías y combinaciones presentes;
- hacemos que los totales correspondan al tamaño real de `registros`.

La llamada preparada trabaja con `pacientes`, sea el millón o el modo de 10,000.


In [9]:
def construir_indices(registros) -> tuple[dict, dict, dict]:
    """Construye tres índices mediante un único recorrido externo."""
    # Resolvemos el contrato y sus invariantes.
    enfermedades = {}
    sintomas = {}
    por_provincias =  {}
    for registro in registros:
        enfermedad = registro["enfermedad"]
        provincia = registro['provincia']
        
        enfermedades[enfermedad] = enfermedades.get(enfermedad , 0) + 1
        
        for sintoma in registro["sintomas"]:
            sintomas[sintoma] = sintomas.get(sintoma, 0) + 1

        if provincia not in por_provincias:
            por_provincias[provincia] = {}
        mapa_provincia = por_provincias[provincia]
        mapa_provincia[enfermedad] = mapa_provincia.get(enfermedad, 0) + 1
    return enfermedades, sintomas, por_provincias

conteo_enfermedades, conteo_sintomas, conteo_por_provincia = construir_indices(pacientes)
verificar_contador(conteo_enfermedades, pacientes, "enfermedad", "enfermedades del conjunto")
verificar_contador(conteo_sintomas, pacientes, "sintomas", "síntomas del conjunto", campo_lista=True)
verificar_mapa_anidado(
    conteo_por_provincia, pacientes, "provincia", "enfermedad", "mapa provincial del conjunto"
)

print(conteo_por_provincia)

Correcto: enfermedades del conjunto contiene 40 claves y suma 1,000,000.
Correcto: síntomas del conjunto contiene 64 claves y suma 3,000,000.
Correcto: mapa provincial del conjunto contiene 7 grupos y suma 1,000,000.
{'Alajuela': {'alzheimer': 3596, 'arritmia': 3680, 'hepatitis': 3567, 'otitis': 3655, 'gota': 3505, 'osteoporosis': 3516, 'alergia': 3563, 'dermatitis': 3506, 'hipertiroidismo': 3585, 'psoriasis': 3557, 'lupus': 3562, 'asma': 3477, 'esclerosis múltiple': 3511, 'epilepsia': 3543, 'lumbalgia': 3553, 'varicela': 3509, 'sinusitis': 3487, 'faringitis': 3504, 'migraña': 3606, 'hipertensión': 3582, 'anemia': 3601, 'colitis': 3533, 'hipotiroidismo': 3583, 'diabetes': 3635, 'tiroiditis': 3677, 'trastorno bipolar': 3573, 'úlcera péptica': 3553, 'obesidad': 3667, 'insuficiencia renal': 3637, 'fibromialgia': 3577, 'artritis': 3528, 'ansiedad generalizada': 3592, 'parkinson': 3649, 'cistitis': 3638, 'gastritis': 3552, 'gripe': 3586, 'bronquitis': 3620, 'conjuntivitis': 3553, 'depresión

## Actividad 7 — Recorremos un diccionario anidado 

Ahora producimos código directamente sobre el índice. Construimos `totales_por_provincia` con **dos ciclos `for` explícitos**: recorremos `conteo_por_provincia.items()` y, dentro, recorremos `mapa.items()`. Sumamos las cantidades internas y guardamos un total verificable por provincia.


In [10]:
totales_por_provincia = {}

# Recorremos ambos niveles con dos ciclos for y acumulamos cada total provincial.
for provincia, mapa in conteo_por_provincia.items():
    totales_provincia = 0
    for enfermedad, cantidad in mapa.items():
        totales_provincia += cantidad
    totales_por_provincia[provincia] = totales_provincia

print(totales_por_provincia)        
        

verificar_contador(
    totales_por_provincia, pacientes, "provincia", "totales calculados por provincia"
)


{'Alajuela': 143056, 'San José': 143581, 'Heredia': 142439, 'Cartago': 142818, 'Limón': 143092, 'Puntarenas': 142351, 'Guanacaste': 142663}
Correcto: totales calculados por provincia contiene 7 claves y suma 1,000,000.


## Actividad 8 — Fijamos máximos y masificamos consultas 

Construimos ambos contratos sin `max()`, `sorted()` ni `lambda`:

```python
obtener_maximo(conteo) -> tuple[str, int]
obtener_maximos_por_grupo(mapa_anidado) -> dict[str, tuple[str, int]]
```

Después consultamos los índices, no la lista `pacientes`. Cuando un conteo está vacío, comunicamos el problema con `ValueError`.


In [11]:
def obtener_maximo(conteo) -> tuple[str, int]:
    """Devuelve (clave, cantidad) para el mayor valor del diccionario."""
    pass


def obtener_maximos_por_grupo(mapa_anidado) -> dict[str, tuple[str, int]]:
    """Calcula una tupla máxima para cada grupo."""
    pass


enfermedad_mas_frecuente = obtener_maximo(conteo_enfermedades)
top_enfermedad_por_provincia = obtener_maximos_por_grupo(conteo_por_provincia)
casos_diabetes_san_jose = conteo_por_provincia["San José"].get("diabetes", 0)

verificar_maximo(enfermedad_mas_frecuente, conteo_enfermedades, "máximo global")
for provincia, resultado in top_enfermedad_por_provincia.items():
    verificar_maximo(resultado, conteo_por_provincia[provincia], f"máximo de {provincia}")

print("Diabetes en San José:", casos_diabetes_san_jose)


AssertionError: Revisamos máximo global: debe ser una tupla de dos elementos.

## Actividad 9 — Resolvemos el reto individual 

Sin plantilla de algoritmo, construimos:

```python
construir_medicamentos_por_enfermedad(registros) -> dict
```

Requisitos:

1. producimos `{enfermedad: {medicamento: cantidad}}`;
2. procesamos todo el conjunto configurado;
3. calculamos el medicamento más frecuente de cada enfermedad como tupla;
4. consultamos la tupla correspondiente a `diabetes`;
5. comprobamos el resultado con las llamadas breves ya preparadas.


In [ ]:
def construir_medicamentos_por_enfermedad(registros) -> dict:
    """Agrupa los medicamentos por enfermedad y cuenta sus apariciones."""
    # Diseñamos y resolvemos el reto abierto.
    pass


medicamentos_por_enfermedad = construir_medicamentos_por_enfermedad(pacientes)
top_medicamento_por_enfermedad = obtener_maximos_por_grupo(medicamentos_por_enfermedad)
medicamento_principal_diabetes = top_medicamento_por_enfermedad["diabetes"]

verificar_mapa_anidado(
    medicamentos_por_enfermedad,
    pacientes,
    "enfermedad",
    "medicamento",
    "medicamentos por enfermedad",
)
for enfermedad, resultado in top_medicamento_por_enfermedad.items():
    verificar_maximo(resultado, medicamentos_por_enfermedad[enfermedad], f"medicamento de {enfermedad}")
print("Medicamento principal para diabetes:", medicamento_principal_diabetes)


## Actividad 10 — Cerramos con el ticket de salida 

Redactamos cuatro respuestas breves con nuestras propias palabras: justificamos el diccionario, la tupla, el ciclo anidado y la ventaja de consultar un índice masivo.


In [ ]:
respuesta_diccionario = ""
respuesta_tupla = ""
respuesta_ciclo_anidado = ""
respuesta_consulta_masiva = ""

print("Elegimos un diccionario porque", respuesta_diccionario)
print("Elegimos una tupla porque", respuesta_tupla)
print("Usamos un ciclo anidado porque", respuesta_ciclo_anidado)
print("Consultamos el índice masivo porque", respuesta_consulta_masiva)


## Extensión opcional

Solo después de dominar el patrón manual, reproducimos el conteo de enfermedades con `Counter` y obtenemos las cinco frecuencias mayores con `sorted()` y `lambda`. Comparamos el resultado con `conteo_enfermedades`.

Esta extensión permanece sin resolver y no forma parte de la producción obligatoria.


In [ ]:
# Opcional: resolvemos aquí con Counter, sorted() y lambda.
